# Which Power Projects Actually Get Built?
## Development Risk in the Middle East and Beyond

*ENSAE Paris, Python for Data Science, 2026-2027*
*Sylvain Topeza, Maxime Rolland, Arcadius Kouwagou*

---

**Summary.** Governments and companies announce far more power plants than they build.
Using Global Energy Monitor's asset-level data on about 180,000 power units worldwide,
enriched with World Bank country data, we measure how much announced capacity is
abandoned rather than built, and what explains the difference.

**How to run this notebook.** Run all cells from top to bottom (about 2 minutes the first
time, while the raw data is downloaded and cached). The notebook runs on any computer with
Python 3.11+ and an internet connection. Setup instructions are in the
`README.md`. The code is organised in modules in `src/`; this notebook only calls them.

In [1]:
import warnings

import pandas as pd
from IPython.display import display

from src import clean, config, indicators, load, model, plots, worldbank

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:,.2f}".format)

# 1. Introduction

### Announced projects and built projects

Energy news is full of announcements: a 5 GW solar park here, a new gas plant there,
national targets of tens of gigawatts. Many of these projects will never be built.
Permits are refused, financing falls through, a tender is cancelled, or a government
changes its priorities.

For anyone who finances or invests in infrastructure, this gap is central. A developer's
value depends on how much of its pipeline will actually reach construction. A government's
energy plan is only as credible as its ability to turn announcements into plants. This
uncertainty is called **development risk**: the risk that a project dies before it is built.

### Why the Middle East

The Middle East is a striking case for two reasons. First, the Gulf states buy electricity
through a single state-owned buyer, which runs tenders and signs long-term purchase
contracts with private developers. This model is very different from the open markets of
Europe or the United States, where developers propose projects and compete for permits,
grid connections and customers. Second, Gulf companies such as ACWA Power (Saudi Arabia) and
Masdar (Abu Dhabi) have become major international developers. Following the same companies
at home and abroad lets us ask whether development risk comes from the developer or from the
market it operates in.

### Problem statement

> **Which power projects actually get built, and why?**

We answer three questions:

1. **How much of what is announced gets built?** We measure abandonment rates worldwide and
   by technology.
2. **Where is development risk highest?** We compare countries, with a focus on the
   Middle East, and relate the results to country characteristics from the World Bank.
3. **Is it the market or the developer?** We compare Gulf developers at home and abroad,
   benchmark them against global peers (ENGIE, EDF, Ørsted, Brookfield, etc.) and estimate
   a logistic regression that separates technology, country and developer effects.

# 2. Data: collection and cleaning

### 2.1 Sources

| Source | Content | Collection |
|---|---|---|
| **Global Integrated Power Tracker**, Global Energy Monitor, September 2026 | One row per power unit (183,404 rows): technology, capacity, status, start year, owners, location | Excel file mirrored, unchanged, on a public URL of our SSP Cloud storage (CC BY 4.0), downloaded by the code |
| **World Bank API** | Income group, GDP per capita and Regulatory Quality (Worldwide Governance Indicators) for each country | Automated queries of the public API, with pagination |
| **ISO country codes** (datasets/country-codes, GitHub) | Country names and ISO codes | CSV read from a public URL; used to join the two sources above |
| **Sponsor dictionary** (`data/sponsors.csv`) | 30 companies, their name patterns and home country | Built by hand from public information |

Global Energy Monitor (GEM) is a non-profit that tracks energy infrastructure worldwide
from public sources: regulators, company reports, local press. The tracker covers power
plants above size thresholds that vary by technology and sometimes by country (for
instance, coal units from 30 MW; in this release, the smallest solar and wind units
listed are 1 MW). Its key feature for us is the **status** of each
unit: from *announced* to *operating*, including *cancelled* and *shelved* projects.
Abandoned projects are rarely kept in other databases, which only list plants that exist.

In [2]:
gem = load.load_gem()
print(f"{gem.shape[0]:,} power units, {gem.shape[1]} columns")

183,404 power units, 54 columns


### 2.2 A first look at the raw data

Each row is a **unit**: a generating unit of a plant or a phase of a solar or wind
project. A plant can therefore span several rows. The columns we use are shown below.

In [3]:
units_raw = clean.select_columns(gem)
display(units_raw.sample(5, random_state=config.RANDOM_SEED)
        .astype({"start_year": "Int64"}))

,unit_id,project,type,country,region,subregion,capacity_mw,status,start_year,owners,parents,latitude,longitude
182059,G100001053795,Purti Sugar power station,bioenergy,India,Asia,Southern Asia,24.00,operating,<NA>,<NA>,<NA>,20.80,79.06
159533,G100000110058,Thap Sakae power station,coal,Thailand,Asia,South-eastern Asia,800.00,cancelled,2025,Electricity Generating Authority of Thailand [...,Electricity Generating Authority of Thailand [...,11.50,99.62
114160,G100001005092,Jiangxi Chongyi Guantian wind farm,wind,China,Asia,Eastern Asia,50.00,pre-construction,<NA>,Jiangxi Datang International New Energy Co Ltd,<NA>,25.58,114.17
20520,G100001014547,Lianhua County Huida Industrial Lianhua County...,utility-scale solar,China,Asia,Eastern Asia,2.60,operating,2016,<NA>,<NA>,27.23,113.96
55089,G100000825957,Belpasso Catania solar farm I,utility-scale solar,Italy,Europe,Southern Europe,2.00,operating,<NA>,<NA>,<NA>,37.50,15.09


In [4]:
status_table = (
    units_raw
    .groupby("status")
    .agg(units=("unit_id", "count"), capacity_gw=("capacity_mw", "sum"))
    .assign(capacity_gw=lambda df: df["capacity_gw"] / 1000,
            share_of_capacity=lambda df: df["capacity_gw"] / df["capacity_gw"].sum())
    .sort_values("capacity_gw", ascending=False)
)
display(status_table.style.format(
    {"units": "{:,.0f}", "capacity_gw": "{:,.0f}", "share_of_capacity": "{:.1%}"}))

,units,capacity_gw,share_of_capacity
status,,,
operating,"128,100","8,719",38.5%
pre-construction,"21,923","3,598",15.9%
announced,"7,578","3,166",14.0%
cancelled,"7,209","2,827",12.5%
construction,"6,235","1,732",7.6%
cancelled - inferred 4 y,"2,989",758,3.3%
retired,"4,389",756,3.3%
shelved - inferred 2 y,"2,809",588,2.6%
shelved,"1,468",358,1.6%


The tracker lists about 22,600 GW of capacity. Operating plants account for 38% of it.
**Projects that are not yet built (announced, pre-construction, construction) total about
8,500 GW**, as much as the whole operating fleet. And about 4,500 GW of projects were
cancelled or shelved. This is the raw material of our question.

Two statuses need care. *Cancelled - inferred 4 y* and *shelved - inferred 2 y* mean that
GEM found no news of the project for four (or two) years and classified it as abandoned
without an official announcement. We treat them as abandoned, and check in section 4 that
our results hold without them.

In [5]:
missing = (
    units_raw.isna().mean()
    .rename("share_missing")
    .to_frame()
    .sort_values("share_missing", ascending=False)
)
display(missing.style.format("{:.1%}"))

,share_missing
parents,81.7%
owners,50.6%
start_year,29.7%
project,0.0%
unit_id,0.0%
region,0.0%
country,0.0%
type,0.0%
subregion,0.0%
status,0.0%


Missing values are concentrated in three columns:

- **Parent companies** (`parents`) are only reported for coal, gas and some other
  thermal plants, where GEM links each owner to its ultimate parent.
- **Owners** (`owners`) are missing for half of the units, but these are mostly small
  solar projects in Europe and Asia: owners are known for 89% of the capacity worldwide,
  and for over 90% of the capacity in the Middle East.
- **Start year** is missing for most projects that are not yet operating, which is
  expected: they have no start date yet. It is known for 90% of operating units.

These gaps shape our method: we cannot follow projects over time (no announcement date),
and our analysis of owners leaves out many small projects.

### 2.3 Cleaning the unit table

The function `clean.build_units` applies five steps, each in its own function:

1. **Select and rename** the useful columns (snake_case names).
2. **Group the ten statuses into four stages**: *pipeline* (announced and pre-construction),
   *construction*, *operating* and *abandoned* (cancelled and shelved, including inferred
   statuses). Retired and mothballed units go into *other*. The mapping is defined in
   `config.STATUS_TO_STAGE`; the function raises an error if GEM introduces a new status.
3. **Add readable labels**: technology names, and a regional grouping (GCC, rest of the
   Middle East, rest of the world).
4. **Add ISO country codes**, needed to join the World Bank data.
5. **Flag the units that reached an outcome** (defined in 2.4).

We also drop two units with a capacity of zero and check that unit identifiers are unique.

In [6]:
country_codes = load.load_country_codes()
units = clean.build_units(gem, country_codes)

stage_table = (
    units.groupby("stage")
    .agg(units=("unit_id", "count"), capacity_gw=("capacity_mw", "sum"))
    .assign(capacity_gw=lambda df: df["capacity_gw"] / 1000)
    .loc[["pipeline", "construction", "operating", "abandoned", "other"]]
)
display(stage_table.style.format({"units": "{:,.0f}", "capacity_gw": "{:,.0f}"}))

,units,capacity_gw
stage,,
pipeline,"29,500","6,764"
construction,"6,235","1,732"
operating,"128,100","8,719"
abandoned,"14,474","4,531"
other,"5,093",896


**Joining country codes.** GEM uses short English country names ("DR Congo", "Türkiye",
"United States"), which do not always match the names of the ISO reference table. We
match them first on the CLDR display names, then on the official names, and fix the 20
remaining cases by hand (`config.COUNTRY_NAME_FIXES`). The check below confirms that every
unit has a country code.

In [7]:
unmatched = units.loc[units["iso3"].isna(), "country"].unique()
print(f"Countries without an ISO code: {len(unmatched)}")
print(f"Countries in the data: {units['country'].nunique()}")

Countries without an ISO code: 0
Countries in the data: 226


### 2.4 Measuring development risk: the outcome sample

GEM gives no announcement date, so we cannot follow a cohort of projects from
announcement to completion. We use the following approach instead.

A project **reached an outcome** if it was either **abandoned** (cancelled or shelved) or
**built** (under construction, or in operation since 2010). Projects still in the pipeline
have no outcome yet, so they are left out. The **abandonment rate** of a group is then:

$$\text{abandonment rate} = \frac{\text{abandoned MW}}{\text{abandoned MW} + \text{built MW}}$$

Why only plants built since 2010? Including every operating plant, some built in the
1950s, would compare decades of successes with the much shorter list of failures that
public sources remember, and make the rate artificially low.

**This cut-off has a limit that we must state clearly: it applies to built projects only.**
Abandoned projects almost never have a date (only 3% do), so all of them enter the
calculation, whatever their era. Old cancellations are thus compared with recent
constructions. This inflates the rates of technologies with a long history of
cancellations: for instance, the 185 GW of abandoned nuclear capacity in the United States
largely date back to the 1970s and 1980s. The rates should therefore be read as orders of
magnitude, and comparisons between technologies with caution; comparisons between
countries and between developers, which mostly concern recent projects, are less affected.

A second choice is optimistic: projects under construction are counted as built, although
some will still be abandoned. Section 4 gives the rates without them.

Rates are **weighted by capacity**: a 2 GW project weighs a hundred times more than a 20 MW
one. This reflects what matters to a grid operator or an investor. Counted in units, the
rate would be only 12%, because most units are small solar farms that are rarely abandoned.

In [8]:
sample_share = units["in_outcome_sample"].mean()
print(f"Units in the outcome sample: {units['in_outcome_sample'].sum():,} "
      f"({sample_share:.0%} of all units)")

Units in the outcome sample: 116,296 (63% of all units)


### 2.5 Who owns what: parsing the ownership fields

Ownership is stored as text, with all owners of a unit in one cell, separated by
semicolons, and their share in brackets when known. The function `clean.parse_owners`
extracts each owner and its share with a regular expression:

In [9]:
example = "Saudi Electricity Co [50%]; ACWA Power Co [30%]; Korea Electric Power Corp [20%]"
clean.parse_owners(example)

[('Saudi Electricity Co', 50.0),
 ('ACWA Power Co', 30.0),
 ('Korea Electric Power Corp', 20.0)]

When no share is reported, capacity is split equally between owners; when only some
shares are reported, owners without a share get zero, so that the total never exceeds 100%.
We use the **ultimate parent** when GEM reports it (coal and gas plants), and the direct
owner otherwise. `clean.build_ownership` then turns the unit table into a **long table with
one row per unit and owner**, and computes each owner's **equity capacity** (capacity ×
share). The table below shows how often shares are actually reported.

In [10]:
ownership = clean.build_ownership(units)

coverage = (
    units.assign(has_owner=units["owners"].notna() | units["parents"].notna())
    .groupby("technology")
    .agg(units=("unit_id", "count"), share_with_owner=("has_owner", "mean"))
    .join(ownership.groupby("technology")["share_reported"].mean()
          .rename("share_of_owner_records_with_percentages"))
    .sort_values("units", ascending=False)
)
display(coverage.style.format({"units": "{:,.0f}", "share_with_owner": "{:.0%}",
                               "share_of_owner_records_with_percentages": "{:.0%}"}))

,units,share_with_owner,share_of_owner_records_with_percentages
technology,,,
Solar,"103,940",23%,75%
Wind,"35,089",70%,78%
Gas & oil,"15,201",99%,100%
Coal,"14,674",100%,100%
Hydropower,"7,023",87%,73%
Bioenergy,"4,815",81%,100%
Nuclear,"1,825",89%,97%
Geothermal,835,85%,66%


Owners are known for almost all thermal plants, but for only a quarter of solar units
(mostly small projects in Europe and Asia). When owners are listed, their shares are
reported in most cases: always for coal and gas, and for about three quarters of solar and
wind ownership records. The equal split is therefore used for a minority of records.

### 2.6 Grouping companies: the sponsor dictionary

A company appears under many names: subsidiaries, project companies, spelling variants.
To follow **30 large sponsors** (12 from the Gulf, 18 global peers), we wrote a dictionary
(`data/sponsors.csv`) with a regular expression per company, and its home country. Two
recent acquisitions not yet reflected in GEM are added by hand: Terna Energy (Greece) and
Saeta Yield (Spain), both bought by Masdar in 2024.

Checking the matches by eye was essential. Our first version contained several false
positives: the pattern *Nebras* (Qatar's power investor) also captured the **Nebraska**
Public Power District; *Neoen* captured **Neoenergia**, Iberdrola's Brazilian subsidiary;
*Mitsui* and *Sumitomo* captured unrelated banks, cement and real-estate companies. Nebras's
operating capacity abroad dropped from 5.3 GW to 1.4 GW once American utilities were removed.
The table below lists, for each group, the largest raw names it now captures.

In [11]:
sponsors = load.load_sponsors()
ownership = clean.assign_sponsor_groups(ownership, sponsors)
display(clean.matched_owner_names(ownership))

,sponsor_group,n_names,largest_names
0,ACWA Power,1,ACWA Power Co
1,AMEA Power,1,AMEA Power
2,Al Nowais,1,Al Nowais Investments LLC
3,Alfanar,2,Alfanar Group; Alfanar Energía España SL
4,Aljomaih,2,Aljomaih Energy & Water Co Ltd; Aljomaih Holdi...
5,Brookfield,15,Brookfield Renewable Partners LP; Brookfield C...
6,EDF,24,EDF Renewables; EDF Energy Ltd; EDF Energy (UK...
7,EDP,12,EDPR Renewable Energy Sources of Greece 1 Sole...
8,ENGIE,22,ENGIE SA; Engie Brasil Energia SA; ENGIE Energ...
9,Enel,24,Enel SpA; Enel Green Power SpA; Enel Generació...
